In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%cd /content/drive/MyDrive/Colab Notebooks/3 - AI process/Vision transfromers

/content/drive/MyDrive/Colab Notebooks/3 - AI process/Vision transfromers


In [4]:
!pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 77.7 MB/s eta 0:00:00


In [7]:
# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
ERROR: This version requires zstd for extraction. Please install zstd and try again:
  - Debian/Ubuntu: sudo apt-get install zstd
  - RHEL/CentOS/Fedora: sudo dnf install zstd
  - Arch: sudo pacman -S zstd


In [10]:
# Install zstd, a dependency for Ollama installation
!sudo apt-get update && sudo apt-get install -y zstd

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [101 kB]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,845 kB]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,125 kB]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:13 http://archive.ubuntu.com/ubuntu

In [11]:
# Re-install Ollama after installing zstd
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [12]:
# Re-start Ollama server in the background
!nohup ollama serve > ollama.log 2>&1 &

In [13]:
# Re-pull the llama2 model
!ollama pull llama2

In [15]:
!apt-get update && apt-get install -y espeak
!pip install gradio ultralytics langchain_ollama pyttsx3

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [1]:
import os
import time
import cv2
import requests
import pyttsx3
import threading
import torch
import random
import numpy as np
import gradio as gr
from collections import deque, defaultdict
from ultralytics import YOLO
from langchain_ollama import OllamaLLM

# --- 1. CROSS-PLATFORM HARDWARE ACCELERATION ENGINE ---
def detect_device_hardware():
    if torch.cuda.is_available():
        return "cuda", f"CUDA ({torch.cuda.get_device_name(0)})"
    return "cpu", "CPU Engine (Fallback)"

def load_model_on_device(model_path, device_target):
    print(f"📦 Initializing model on hardware target: {device_target}...")
    try:
        model = YOLO(model_path)
        if str(device_target) != "cpu":
            try:
                model.to(device_target)
            except Exception as e:
                print(f"⚠️ Hardware transfer failed ({e}). Falling back to CPU.")
                model.to("cpu")
                return model, "cpu", "CPU Engine (Fallback)"
        return model, device_target, None
    except Exception as e:
        print(f"⚠️ Model load error ({e}). Falling back to CPU.")
        model = YOLO(model_path).to("cpu")
        return model, "cpu", "CPU Engine (Fallback)"

# --- 2. GLOBAL STATE & PATHS SETUP ---
script_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else '/content/drive/MyDrive/Colab Notebooks/3 - AI process/Vision transfromers'
model_path = os.path.join(script_dir, "fish_model.pt")
video_path = os.path.join(script_dir, "dehazed_videos_outputdehazed_main_2X_60fps.mp4")

target_dev, hw_desc = detect_device_hardware()
llm = OllamaLLM(model="llama3")
model, active_device, fallback_desc = load_model_on_device(model_path, target_dev)

# Gradio State Variables (Replacing OpenCV Key Listeners)
action_trigger = None
user_chat_buffer = ""

current_fx_idx = 0
vision_fx_names = ["NORMAL", "THERMAL", "NIGHT VISION", "SONAR EDGE", "CLAHE"]
selectable_filters = ["ALL OBJECTS", "Salmo trutta", "FISH ONLY"]
current_filter_idx = 0

show_heatmap = True
show_motion_vectors = True
show_sonar_grid = False
show_pip_zoom = True
conf_threshold = 0.40
show_water_gif = False
language_mode = "EN"

ACTIVE_COMM = None
pauly_ui_alpha = 0.0
pauly_fade_state = "IDLE"
pauly_reading_timer = 0
pauly_active_dialogue = ""

locked_target = None

# Asset Loaders
johnny_img_rgba = None
scuba_pauly_img = None
water_gif_cap = None
GLOBAL_SPECIES_IMAGES = {}
recent_gbif_species = deque(maxlen=4)

# --- 3. CORE PROCESSING LOGIC (Stripped of OpenCV UI Code) ---
def fetch_gbif_image(species_name):
    search_term = species_name.replace(" ", "%20")
    gbif_url = f"https://api.gbif.org/v1/occurrence/search?scientificName={search_term}&mediaType=StillImage&limit=5"
    try:
        res = requests.get(gbif_url, headers={"User-Agent": "SpreewaldCyberdeck/1.0"}, timeout=4)
        if res.status_code == 200:
            for occ in res.json().get("results", []):
                for media in occ.get("media", []):
                    img_url = media.get("identifier")
                    if img_url:
                        img_res = requests.get(img_url, timeout=4)
                        if img_res.status_code == 200:
                            img_arr = np.frombuffer(img_res.content, np.uint8)
                            decoded = cv2.imdecode(img_arr, cv2.IMREAD_COLOR)
                            if decoded is not None:
                                return cv2.resize(cv2.cvtColor(decoded, cv2.COLOR_BGRA2BGR) if decoded.shape[2]==4 else decoded, (380, 220))
    except Exception: pass
    return None

def async_preload_species_image(species_name):
    if species_name in GLOBAL_SPECIES_IMAGES: return
    img = fetch_gbif_image(species_name)
    GLOBAL_SPECIES_IMAGES[species_name] = img if img is not None else "FAILED"

def ask_dr_pauly(species_name, user_question=None):
    prompt = f"You are Dr. Daniel Pauly. Explain the fish {species_name}. Keep it strictly under 40 words."
    try: return llm.invoke(prompt)
    except: return f"Ah, hello! Dr. Pauly here. Fascinating {species_name} specimen."

def background_speak(text):
    try:
        engine = pyttsx3.init()
        engine.say(text)
        engine.runAndWait()
    except Exception as e:
        print(f"Colab TTS Error (Check espeak): {e}")

def trigger_pauly_call(target_species, user_question=None):
    global pauly_active_dialogue, pauly_fade_state, ACTIVE_COMM
    if target_species not in GLOBAL_SPECIES_IMAGES:
        threading.Thread(target=async_preload_species_image, args=(target_species,), daemon=True).start()

    pauly_active_dialogue = ask_dr_pauly(target_species, user_question)
    pauly_fade_state = "FADE_IN"
    ACTIVE_COMM = "PAULY"
    threading.Thread(target=background_speak, args=(pauly_active_dialogue,), daemon=True).start()

# --- 4. GRADIO UI CALLBACKS (Updates Global State) ---
def set_action(trigger_name):
    global action_trigger
    action_trigger = trigger_name

def submit_chat(text):
    global user_chat_buffer, action_trigger
    if text.strip():
        user_chat_buffer = text
        action_trigger = "submit_chat"
    return ""

def auto_lock_target():
    global action_trigger
    action_trigger = "auto_lock"

# --- 5. THE LIVE VIDEO GENERATOR (Yields frames to Gradio) ---
def generate_video_frames():
    global action_trigger, current_filter_idx, current_fx_idx, show_motion_vectors
    global show_heatmap, show_sonar_grid, show_pip_zoom, language_mode, show_water_gif
    global conf_threshold, chat_mode_active, ACTIVE_COMM, pauly_fade_state, locked_target
    global pauly_reading_timer, pauly_active_dialogue, pauly_ui_alpha

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        yield np.zeros((480, 640, 3), dtype=np.uint8)
        return

    frame_counter = 0
    active_boxes = []

    while cap.isOpened():
        success, raw_frame = cap.read()
        if not success: break
        frame_counter += 1

        # Process Gradio Actions
        if action_trigger:
            if action_trigger == "toggle_filter": current_filter_idx = (current_filter_idx + 1) % len(selectable_filters)
            elif action_trigger == "toggle_fx": current_fx_idx = (current_fx_idx + 1) % len(vision_fx_names)
            elif action_trigger == "toggle_vec": show_motion_vectors = not show_motion_vectors
            elif action_trigger == "toggle_heat": show_heatmap = not show_heatmap
            elif action_trigger == "toggle_sonar": show_sonar_grid = not show_sonar_grid
            elif action_trigger == "toggle_pip": show_pip_zoom = not show_pip_zoom
            elif action_trigger == "toggle_lang": language_mode = "DE" if language_mode == "EN" else "EN"
            elif action_trigger == "toggle_gif": show_water_gif = not show_water_gif
            elif action_trigger == "conf_inc": conf_threshold = min(0.95, conf_threshold + 0.05)
            elif action_trigger == "conf_dec": conf_threshold = max(0.15, conf_threshold - 0.05)
            elif action_trigger == "call_pauly":
                target = locked_target["species"] if locked_target else "Salmo trutta"
                trigger_pauly_call(target)
            elif action_trigger == "submit_chat":
                target = locked_target["species"] if locked_target else "Salmo trutta"
                trigger_pauly_call(target, user_question=user_chat_buffer)
                user_chat_buffer = ""
            elif action_trigger == "auto_lock":
                # Fallback for mouse click: locks onto the first active box
                if active_boxes:
                    box, tid, sp_n, conf, cx, cy = active_boxes[0]
                    locked_target = {"id": tid, "species": sp_n, "conf": conf, "lost_frames": 0, "last_center": (cx,cy)}
            action_trigger = None

        # Pauly Fade State Machine
        if pauly_fade_state == "FADE_IN":
            pauly_ui_alpha = min(1.0, pauly_ui_alpha + 0.05)
            if pauly_ui_alpha >= 1.0: pauly_fade_state = "ACTIVE"
        elif pauly_fade_state == "ACTIVE":
            # For Colab, we simulate reading time instead of audio lock
            pauly_reading_timer = 150
            pauly_fade_state = "READING_GRACE"
        elif pauly_fade_state == "READING_GRACE":
            pauly_reading_timer -= 1
            if pauly_reading_timer <= 0: pauly_fade_state = "FADE_OUT"
        elif pauly_fade_state == "FADE_OUT":
            pauly_ui_alpha = max(0.0, pauly_ui_alpha - 0.05)
            if pauly_ui_alpha <= 0.0:
                pauly_fade_state = "IDLE"
                ACTIVE_COMM = None

        frame = raw_frame.copy()

        # YOLO Tracking
        results = model.track(frame, persist=True, tracker="botsort.yaml", device=active_device, conf=conf_threshold, verbose=False)
        active_boxes.clear()

        if results[0].boxes is not None and results[0].boxes.id is not None:
            boxes = results[0].boxes.xyxy.cpu().numpy()
            track_ids = results[0].boxes.id.cpu().numpy()
            confidences = results[0].boxes.conf.cpu().numpy()
            names = results[0].names
            class_indices = results[0].boxes.cls.cpu().numpy()

            for box, tid, cls_idx, conf in zip(boxes, track_ids, class_indices, confidences):
                sp_n = names[int(cls_idx)]
                x1, y1, x2, y2 = map(int, box)
                cx, cy = (x1+x2)//2, (y1+y2)//2
                active_boxes.append((box, tid, sp_n, conf, cx, cy))

                # Draw Box
                is_locked = (locked_target and locked_target['id'] == tid)
                color = (255, 255, 0) if is_locked else (0, 255, 255)
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(frame, f"ID:{tid} {sp_n} {conf:.2f}", (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

        # Draw Telemetry overlays directly onto the video frame for Gradio
        if ACTIVE_COMM == "PAULY" and pauly_ui_alpha > 0:
            overlay = frame.copy()
            cv2.rectangle(overlay, (10, 10), (400, 150), (15, 20, 30), -1)
            cv2.putText(overlay, f"DR. PAULY [{language_mode}]:", (20, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

            # Simple text wrap for Colab display
            words = pauly_active_dialogue.split()
            y_offset = 60
            line = ""
            for word in words:
                if len(line) + len(word) > 45:
                    cv2.putText(overlay, line, (20, y_offset), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 150), 1)
                    y_offset += 20
                    line = word + " "
                else:
                    line += word + " "
            cv2.putText(overlay, line, (20, y_offset), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 150), 1)

            frame = cv2.addWeighted(overlay, pauly_ui_alpha, frame, 1 - pauly_ui_alpha, 0)

        # Convert BGR to RGB for Gradio Web UI
        yield cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    cap.release()

# --- 6. GRADIO WEB DASHBOARD LAYOUT ---
with gr.Blocks(theme=gr.themes.Monochrome()) as demo:
    gr.Markdown("## ⚡ SPREEWALD CYBERDECK (Google Colab Edition) ⚡")

    with gr.Row():
        with gr.Column(scale=3):
            video_feed = gr.Image(label="Live Tracking Feed", interactive=False)

        with gr.Column(scale=1):
            gr.Markdown("### 🎛️ Telemetry Controls")

            with gr.Row():
                btn_filter = gr.Button("🎯 Cycle Filter [S]")
                btn_fx = gr.Button("👁️ Vision FX [F]")
            with gr.Row():
                btn_heat = gr.Button("🔥 Heatmap [H]")
                btn_zoom = gr.Button("🔍 PiP Zoom [Z]")
            with gr.Row():
                btn_conf_inc = gr.Button("⚙️ Conf [+]")
                btn_conf_dec = gr.Button("⚙️ Conf [-]")

            btn_auto_lock = gr.Button("🔒 Auto-Lock Nearest Target", variant="primary")

            gr.Markdown("### 📡 Comm Links")
            btn_pauly = gr.Button("📞 Call Dr. Pauly [C]")
            btn_johnny = gr.Button("🎸 Johnny Relic [J]")
            btn_lang = gr.Button("🌐 Toggle Language [L]")

            gr.Markdown("### 💬 User Chat Override")
            chat_input = gr.Textbox(label="Message Dr. Pauly", placeholder="Type question here...")
            btn_chat = gr.Button("Send Transmission [Enter]")

    # Event Bindings
    btn_filter.click(fn=lambda: set_action("toggle_filter"))
    btn_fx.click(fn=lambda: set_action("toggle_fx"))
    btn_heat.click(fn=lambda: set_action("toggle_heat"))
    btn_zoom.click(fn=lambda: set_action("toggle_pip"))
    btn_conf_inc.click(fn=lambda: set_action("conf_inc"))
    btn_conf_dec.click(fn=lambda: set_action("conf_dec"))
    btn_auto_lock.click(fn=auto_lock_target)
    btn_lang.click(fn=lambda: set_action("toggle_lang"))
    btn_pauly.click(fn=lambda: set_action("call_pauly"))
    btn_johnny.click(fn=lambda: set_action("trigger_johnny"))
    btn_chat.click(fn=submit_chat, inputs=[chat_input], outputs=[chat_input])
    chat_input.submit(fn=submit_chat, inputs=[chat_input], outputs=[chat_input])

    # Start the generator loop upon load
    demo.load(fn=generate_video_frames, outputs=[video_feed])

# Launch the app inline inside Colab
if __name__ == "__main__":
    demo.launch(debug=True, inline=True, share=True)

ModuleNotFoundError: No module named 'pyttsx3'